# **Imports**


In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Laad en verwerk je data
X_train = pd.read_csv('Datasets/cleaned_X_train2.csv')
y_train = pd.read_csv('Datasets/y_train.csv')

# Zet y_train om naar één enkele kolom van labels (gebruik idxmax als er meerdere kolommen zijn)
y_train = y_train.idxmax(axis=1)

# Codeer de labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Schaal de X_train dataset
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Maak een SMOTE--object aan
smote = SMOTE(random_state=42)

# Pas SMOTE toe op de trainingsgegevens
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train_encoded)

# Toepassen van PCA
pca = PCA(n_components=0.2)  # Verklaar 20% van de variantie
X_train_pca = pca.fit_transform(X_train_balanced)

# Verdeel de data in training en validatie sets
X_train_final, X_val, y_train_final, y_val = train_test_split(X_train_pca, y_train_balanced, test_size=0.2, random_state=42)

# Laad de testset
X_test = pd.read_csv('Datasets/cleaned_X_test2.csv')
y_test = pd.read_csv('Datasets/y_test.csv')

# Imputeer missende waarden
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Zet y_test om naar één enkele kolom van labels
y_test = y_test.idxmax(axis=1)

# Codeer de testlabels op dezelfde manier als de trainingslabels
y_test_encoded = label_encoder.transform(y_test)

# Schaal de X_test dataset met dezelfde scaler die we voor X_train hebben gebruikt
X_test_scaled = scaler.transform(X_test)

# Pas dezelfde PCA-transformatie toe op de testdata
X_test_pca = pca.transform(X_test_scaled)

# Initialiseer de XGBoost classifier
xgb_model = XGBClassifier(random_state=42, objective='multi:softmax', num_class=10, early_stopping_rounds=10)

# Verklaar een kleinere zoekruimte voor hyperparameters
param_dist = {
    'learning_rate': np.linspace(0.01, 0.1, 10),
    'n_estimators': [50, 100],  # Verlaag het aantal bomen om overfitting te verminderen
    'max_depth': [3, 5],  # Beperk de diepte van de bomen
    'subsample': [0.7, 0.8],  # Beperk de subample om overfitting te verminderen
    'colsample_bytree': [0.7, 0.8],  # Verlaag de colsample_bytree om overfitting te verminderen
}

# Gebruik RandomizedSearchCV om hyperparameters te optimaliseren
random_search = RandomizedSearchCV(xgb_model, param_distributions=param_dist, n_iter=5, cv=3, scoring='accuracy', random_state=42)

# Train het model met early stopping
random_search.fit(X_train_final, y_train_final, eval_set=[(X_val, y_val)], verbose=False)

# Toon de beste hyperparameters
print("Beste hyperparameters gevonden door RandomizedSearchCV:")
print(random_search.best_params_)

# Voorspel met het beste model op de validatieset
y_pred = random_search.best_estimator_.predict(X_val)

# Evaluatie op de validatieset
accuracy = accuracy_score(y_val, y_pred)
print(f'Accuracy op de validatieset: {accuracy:.4f}')
print("\nClassification Report op de validatieset:")
print(classification_report(y_val, y_pred))

# Maak voorspellingen op de testset met het beste model
y_test_pred = random_search.best_estimator_.predict(X_test_pca)

# Evaluatie op de testset
test_accuracy = accuracy_score(y_test_encoded, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')
print("\nTest Classification Report:")
print(classification_report(y_test_encoded, y_test_pred))


/home/iyoas/.local/lib/python3.10/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Beste hyperparameters gevonden door RandomizedSearchCV:
{'subsample': 0.8, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.07, 'colsample_bytree': 0.7}
Accuracy op de validatieset: 0.4919

Classification Report op de validatieset:
              precision    recall  f1-score   support

           0       0.43      0.46      0.44        13
           1       0.64      0.56      0.60        16
           2       0.23      0.60      0.33         5
           3       0.32      0.55      0.40        11
           4       0.75      0.46      0.57        13
           5       0.50      0.46      0.48        13
           6       0.30      0.25      0.27        12
           7       0.50      0.45      0.48        11
           8       0.67      0.50      0.57        16
           9       0.75      0.64      0.69        14

    accuracy                           0.49       124
   macro avg       0.51      0.49      0.48       124
weighted avg       0.54      0.49      0.50       124

Te